In [ ]:

# ABLATION 1: V3-MINUS-AUG (ULTRA-LIGHTWEIGHT + PERSISTENT LOADERS)
# Architecture: EfficientNet-B4
# Change: PIL Loading + Persistent Workers to prevent RAM and Indexing hangs.

import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import numpy as np
import sys
import os
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dir = '/content/skin-lesion-classifier/data/model_data/train'
val_dir = '/content/skin-lesion-classifier/data/model_data/val'

#native pyTorch transforms
naked_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#image loader
train_dataset = datasets.ImageFolder(train_dir, transform=naked_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=naked_transforms)

#weighted random sampler
target = np.array(train_dataset.targets)
class_sample_count = np.array([len(np.where(target == t)[0]) for t in np.unique(target)])
sampler_weights = 1. / class_sample_count
samples_weight = np.array([sampler_weights[t] for t in target])
sampler = WeightedRandomSampler(torch.from_numpy(samples_weight), len(samples_weight))

#optimized Data Loaders
#num_workers=2 and persistent_workers=True fixes the "12-minute hang" without crashing RAM.
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

#initialize Model
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=2)
model = model.to(device)

#loss and weight optimizer
loss_weights = 1. / np.sqrt(class_sample_count)
weights_tensor = torch.FloatTensor(loss_weights).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

num_epochs = 30
print(f"Starting Optimized ULTRA-LIGHT Ablation Training for {num_epochs} Epochs...")

for epoch in range(num_epochs):
    #training
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        #immediate micro-purge for RAM
        del images, labels, outputs, loss

    avg_train_loss = running_loss / len(train_loader)

    #validation phase
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            #immediate micro-purge for RAM
            del images, labels, outputs, loss

    #metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f} | Recall: {recall:.4f} | Prec: {precision:.4f}")

    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f'/content/skin-lesion-classifier/models/ablation_minus_aug_epoch_{epoch+1}.pth')

    #macro-purge after every epoch
    gc.collect()
    torch.cuda.empty_cache()

torch.save(model.state_dict(), '/content/skin-lesion-classifier/models/ablation_minus_aug_final.pth')
print("Ablation Training Complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Starting Optimized ULTRA-LIGHT Ablation Training for 30 Epochs...
Epoch [1/30] - Train Loss: 0.3028 | Val Loss: 0.2520 | Val Acc: 0.8931 | Val F1: 0.9064 | Recall: 0.8938 | Prec: 0.9194
Epoch [2/30] - Train Loss: 0.1513 | Val Loss: 0.2298 | Val Acc: 0.9025 | Val F1: 0.9107 | Recall: 0.8578 | Prec: 0.9705
Epoch [3/30] - Train Loss: 0.1022 | Val Loss: 0.1558 | Val Acc: 0.9358 | Val F1: 0.9447 | Recall: 0.9460 | Prec: 0.9434
Epoch [4/30] - Train Loss: 0.0607 | Val Loss: 0.1736 | Val Acc: 0.9416 | Val F1: 0.9502 | Recall: 0.9622 | Prec: 0.9385
Epoch [5/30] - Train Loss: 0.0485 | Val Loss: 0.1303 | Val Acc: 0.9546 | Val F1: 0.9608 | Recall: 0.9604 | Prec: 0.9613
Epoch [6/30] - Train Loss: 0.0348 | Val Loss: 0.1310 | Val Acc: 0.9557 | Val F1: 0.9614 | Recall: 0.9523 | Prec: 0.9706
Epoch [7/30] - Train Loss: 0.0304 | Val Loss: 0.1485 | Val Acc: 0.9531 | Val F1: 0.9597 | Recall: 0.9640 | Prec: 0.9554
Epoch [8/30] - Train Loss: 0.0253 | Val Loss: 0.1846 | Val Acc: 0.9463 | Val F1: 0.9535 | Reca